# API Utility for Fetching Data and Writing to ADLS

This notebook defines a class `APIClient` that facilitates interaction with an API, handling different types of authentication and making GET requests to specified endpoints. It supports Bearer token, API key, and Basic authentication methods.

Additionally, there is a function `write_api_response` to write the API response data into a JSON file at a specified target path in Azure Data Lake Storage (ADLS) Gen2.

## Import necessary packages

In [ ]:
# Import necessary packages
import requests
import json
from typing import Dict, Any, Optional
from requests.auth import HTTPBasicAuth

## API Client class

In [ ]:
# Class for interacting with the API. It handles different types of authentication and makes GET requests to the specified endpoints.
class APIClient:

    # Initializes the APIClient with the base URL, headers, and authentication information.
    def __init__(self, base_url: str, headers: Dict[str, str] = None, auth: Optional[Dict[str, Any]] = None):
        self.base_url = base_url
        self.headers = headers or {}
        self.auth = auth

    # Returns the appropriate authentication headers based on the auth type.
    def _get_auth_headers(self):
        if not self.auth:
            return {}

        auth_type = self.auth.get("type")
        if auth_type == "bearer_token":
            return {"Authorization": f"Bearer {self.auth['access_token']}"}
        elif auth_type == "api_key":
            return {self.auth['key_name']: self.auth['key_value']}
        elif auth_type == "basic":
            return HTTPBasicAuth(self.auth['username'], self.auth['password'])
        else:
            raise ValueError(f"Unsupported authentication type: {auth_type}")

    # Makes a GET request to the specified endpoint with optional parameters.
    def get(self, endpoint: str, params: Dict[str, Any] = None) -> Dict[str, Any]:
        auth_headers = self._get_auth_headers()
        headers = {**self.headers, **auth_headers} if isinstance(auth_headers, dict) else self.headers
        try:
            response = requests.get(f"{self.base_url}/{endpoint}", headers=headers, params=params, auth=auth_headers if not isinstance(auth_headers, dict) else None)
            response.raise_for_status()
            return response.json()
        except requests.exceptions.RequestException as e:
            print(f"Error fetching data from {endpoint}: {e}")
            raise

## Write API response

In [ ]:
# Writes the API response to a JSON file at the specified target path on ADLS Gen2.
def write_api_response(response, target_path):
    try:
        dbutils.fs.put(target_path, json.dumps(response, indent=4), True)
        print(f"Data written to {target_path}")
    except Exception as e:
        print(f"Error writing data to {target_path}: {e}")
        raise